In [ ]:
import torch
from torch.nn.functional import logsigmoid
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
import os, csv

# set a device + specify number of seeds, uniform-mixing & CI target coverage + spread of budgets
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, TAU, ALPHA = 1000, 0.5, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)

# which dataset + method are we running?
dataset, method = "pew", "sai"

# filename for this simulation
fname = f"{dataset}_sai.csv"

# dataframe to store all of our results
columns = ["dataset", "budget", "tau", "method", "seed", "thetahat", "coverage", "width", "n_labels"]
if fname not in os.listdir("../logs"):
    with open(f"../logs/{fname}", "wt") as file:
        file.write(",".join(columns) + "\n")
        
# dummy tensors for brevity
ZERO = torch.tensor(0.0, device=device)

In [ ]:
#### LOAD IN OUR DATA

# load in X_test and y_test + initial weights for OLR
Xs = torch.load(f"../datasets/{dataset}/X_test.pt").to(device) # (N, D)
ys = torch.load(f"../datasets/{dataset}/y_test.pt").to(device) # (N,)
beta_init = torch.load(f"../datasets/{dataset}/beta_init.pt").to(device) # (D,)

# get our finite-population size
N, D = Xs.shape

In [ ]:
#### SEQUENTIAL ACTIVE INFERENCE SAMPLING

# iterate through our budgets
for budget_idx, budget in enumerate(BUDGET_PROPS):
    
    # how many labels can we collect?
    n = int(budget * N)
    
    # Zrnic & Candes budget catch-up heuristic
    GREEDY_STEPS = 100 # for binary classification datasets, evidenced in original paper.
    budget_window = int(GREEDY_STEPS / budget)
    
    #### initialize data structures
    
    # set a seed for reproducibility
    torch.random.manual_seed(858 + budget_idx)

    # how many labels have we used up already
    obs_counts = torch.zeros(size=(N_SEEDS, 1), dtype=torch.float32, device=device)

    # recurring sum (WITHOUT NORMALIZATION) of thetahat (need div by N) and sigmahat^2 (need div by N^2)
    thetahats_unnormalized = torch.zeros(
        size=(N_SEEDS, 1), device=device, dtype=torch.float32) # (N_SEEDS, 1)
    sigmahats_sq_unnormalized = torch.zeros(
        size=(N_SEEDS, 1), device=device, dtype=torch.float32) # (N_SEEDS, 1)
    
    # initialized LR coefficients + let's use identity initial diagonal precisions
    betas = torch.tile(beta_init, (N_SEEDS, 1)) # (N)
    diag_precisions = torch.full(size=(N_SEEDS, D), fill_value=1.0, device=device, dtype=torch.float32) # (N_SEEDS, D)
    
    #### sequential active inference (need loop thru full dataset, yes budget stabilization if "sai")
    for t in tqdm(range(N), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}"):
        
        #### PART 1. computing q_t(j) probabilities + sampling probabilities
        
        # a. compute the uncertainty scores based on the current ML prediction
        logits = betas @ Xs.T # (N_SEEDS, N)
        fxs = torch.sigmoid(logits) # (N_SEEDS, N)

        # b. more stable: min(\sig(z), 1 - \sig(z)) = min(\sig(z), \sig(-z)) = \sig(min(z, -z)) = \sig(-|z|)
        log_u_ts = logsigmoid(-logits.abs())
        
        # c. pi_t = (n/N) * u_{t,t} / E[u_t]
        pi_t = (n * torch.softmax(log_u_ts, dim=-1))[:,t:t+1] # (N_SEEDS, 1)
        
        # d. for sequential: budget-stabilization wrapper (Eqn 8 + Algorithm 2, Zrnic & Candes '24)
        if method == "sai":
            
            # i. budget stabilization wrapper needs "current budget allocation"
            n_delta_ts = ((t+1) * n / N) - obs_counts # (N_SEEDS, 1)
            
            '''
            For the last 100 observations of each budget window, 
            ignore uncertainty and aggressively catch up on unused budget.
            
            See: Section 8.5 in Zrnic & Candes (2024).
            '''
            # ii. arbitrary budget underutilization prevention
            if (t % budget_window) >= (budget_window - GREEDY_STEPS):
                pi_t = n_delta_ts.clamp(min=0.0, max=1.0)

            # iii. otherwise: ordinary Eq. (8) budget stabilization
            else:
                pi_t = torch.minimum(pi_t, n_delta_ts).clamp(min=0.0, max=1.0) # (N_SEEDS, 1)
            
        # e. mixing with uniform -- Section 7.2, Zrnic & Candes '24
        pi_t_tau = ((1.0 - TAU) * pi_t) + (TAU * (n / N)) # (,)

        # f. are we going to query this datapoint or not?
        xi_t = torch.bernoulli(pi_t_tau)
        
        #### PART 2. UPDATING SEQUENTIAL ACTIVE INFERENCE ESTIMATOR + BOOKKEEPING

        # a. update our means and variances
        thetahats_unnormalized += ((xi_t / pi_t_tau) * (ys[t] - fxs[:,t:t+1])) + fxs[:,t:t+1]
        sigmahats_sq_unnormalized += (xi_t * (1.0 - pi_t_tau) / (pi_t_tau ** 2) ) * ((ys[t] - fxs[:,t:t+1]) ** 2)

        # b. did we exhaust budget? also, did we hit a big-residual datapoint?
        obs_counts += xi_t
        
        #### PART 3: updating our online logistic regression
        
        # a. convert xi_t to bool
        xi_t = xi_t.to(bool).squeeze() # (N_NEW,)
        
        # b. fhat * (1 - fhat); note that diag(xx^T) = (x_1^2, \dots, x_D^2)
        w_Is = fxs[:,t:t+1][xi_t] * (1.0 - fxs[:,t:t+1][xi_t]) # (N_FLIP_HEADS, 1)
        x_t = Xs[t] # (D,)
        diag_precisions[xi_t] += w_Is * (x_t ** 2) # (N_SEEDS, D)
        betas[xi_t] += (ys[t] - fxs[:,t:t+1][xi_t]) * x_t / diag_precisions[xi_t] # (N_SEEDS, D)
        
    #### COMPUTING SEQUENTIAL ACTIVE INFERENCE CONFIDENCE INTERVAL AND COVERAGE
    
    # a. dummy tensor for brevity
    ZERO = torch.tensor(0.0, device=device)

    # b. get our mean estimate + variance estimate
    thetahats = thetahats_unnormalized / N
    sigmahats = torch.sqrt(torch.maximum(sigmahats_sq_unnormalized, ZERO) / N)
    sigmahats_bar = sigmahats / torch.sqrt(torch.tensor(N, device=device))

    # c. get our z-score + the lower/upper limits of our interval
    z_score = norm.ppf(1 - (ALPHA / 2))
    lbs = thetahats - z_score * sigmahats_bar
    ubs = thetahats + z_score * sigmahats_bar
    widths = (ubs - lbs).cpu().numpy()
    coverages = ((lbs <= ys.mean(dim=-1, keepdim=True)) & (ys.mean(dim=-1, keepdim=True) <= ubs)).cpu().numpy()
    
    # d. for file writing purposes, convert thetahats + obs_counts to numpy
    thetahats = thetahats.cpu().numpy()
    obs_counts = obs_counts.cpu().numpy()
    
    # e. format results + directly write to file
    results = [
        [dataset, budget, TAU, method, seed, 
         thetahats[seed][0], coverages[seed][0], widths[seed][0], obs_counts[seed][0]
        ] for seed in range(N_SEEDS)]
    with open(f"../logs/{fname}", "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerows(results)